# CSE 4310
# Programming Assignment 1 (P1)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
import math

### Task 1 (20 points) 

Import the "rgb.png" image 

<img src = "Images/rgb.png" style="width:300px;height:200px">

a) Display the Red, Green and Blue color channels separately in a 1X3 plot. Use grayscale colormap for displaying the images (cmap = "gray" in matplotlib)
DONE

b) Convert the image to grayscale using the formula, grayscale = (0.33 * Red + 0.33 * Green + 0.33 * Blue) and display the image.
DONE

c) Apply an appropriate mask on the "rgb.png" image based on pixel intensity values to display the red pixels only.
DONE

d) Convert the "rgb.png" image to HSV, adjust the value/brightness value in the HSV image to 50%, convert the HSV back to RGB, and display the result.
DONE

In [ ]:
imported_rgb = cv2.imread("Images/rgb.png")

########### TASK 1A ###########
red = imported_rgb[:, :, 2]
green = imported_rgb[:, :, 1]
blue = imported_rgb[:, :, 0]

fig, ax = plt.subplots(1, 3)
ax[0].imshow(red, cmap="gray")
ax[0].set_title("Red Channel")

ax[1].imshow(green, cmap="gray")
ax[1].set_title("Green Channel")

ax[2].imshow(blue, cmap="gray")
ax[2].set_title("Blue Channel")

plt.show()

In [ ]:

########### TASK 1B ###########
# For task 1B, we will handle each channel separetely and then calculate respective component, aggregating the sum in a "grayscale" place holder. 
grayscale = np.zeros_like(imported_rgb) 
grayscale = grayscale[:, :]

# Blue channel
grayscale_blue = imported_rgb[:, :, 0] * 0.33
# Green channel
grayscale_green = imported_rgb[:, :, 1] * 0.33
# Red Channel
grayscale_red = imported_rgb[:, :, 2] * 0.33


grayscale = np.rint(grayscale_blue + grayscale_green + grayscale_red).astype(np.uint8) # earlier I did not do the type conversions and did not notice any notable differences



plt.imshow(cv2.cvtColor(grayscale, cv2.COLOR_BGR2RGB))
plt.title("Task 1B")
plt.show()

In [ ]:

########### TASK 1C ###########

# As mentioned in class, mask can be implemented using a "MAX" or some sort of
# threshold. We will use a threshold, s.t. if r pixel value is above 150 (first
# guess), then we keep the pixel. Alternatively, I could have found the max of
# RGB value of the image (probably 255) and just do a check or use some
# standard deviation blah blah.  Anyways, we will just do "fine-tuning" (im
# guessing that 150 threshold is sufficient).

# TLDR: For each pixel, (x and y), compare the R channel to see if the value is above 150. If yes, keep, otherwise set all channels to be 0 for that pixel.grayscale
masked_img = imported_rgb.copy()
threshold = 150 

for row in range(0, 479):
    for col in range(0, 639):
        red_channel_value = masked_img[row][col][2]
        green_channel_value = masked_img[row][col][1]
        blue_channel_value = masked_img[row][col][0]
        if red_channel_value < threshold or red_channel_value <= green_channel_value:
            masked_img[row][col][:] = 0

converted_bgr_to_rgb_task1 = cv2.cvtColor(masked_img, cv2.COLOR_BGR2RGB)

plt.imshow(converted_bgr_to_rgb_task1) 
plt.title("Task 1C")
plt.show() 

In [ ]:

########### TASK 1D ###########

# So we first gotta convert the encoding of the image and store it. We can then
# split the H, S, and V and then get V' which would be half of 255. Clipping
# should be used such that V stays in the range of 0-255. Then, reconstruct the
# image with the V' value.

task1_hsv = cv2.cvtColor(imported_rgb, cv2.COLOR_BGR2HSV)
H, S, V = cv2.split(task1_hsv)

V[:] = int(255 / 2)

#task1_hsv_prime = cv2.merge([H, S, V])

new_image = cv2.cvtColor(cv2.merge([H, S, V]), cv2.COLOR_HSV2RGB)

plt.imshow(new_image)
plt.title("Task 1D")
plt.show()

Task 2

DONE
a) 

You want to extract the green apple from the image above. Apply a suitable mask to extract the green apple from the RGB image. 

You will now do the same thing in HSV space. Covert the image to HSV space and apply a suitable mask to extract the green apple.  

Display the original image, the masked image in RGB space and the masked image in HSV color space in a 1X3 plot. 


DONE
b) Import the "fruits.jpg" image 

<img src = "Images/fruits.jpg" style="width:300px;height:180px">

You want to extract the orange colors from the image above. Apply a suitable mask to extract the orange pixels in RGB color space. 

You will now do the same thing in HSV space. Covert the same image to HSV space and apply a suitable mask to extract the orange color pixels.  

Display the original image, the masked image in RGB space and the masked image in HSV color space in a 1X3 plot. 

In [ ]:

########### TASK 2A ###########

imported_fruit = cv2.imread("Images/apple.jpg")

# Originally I wanted to use the same approach, but I didn't like that it would
# maintain the white background. So instead, we will do a modified version that
# starts off with blacking off the whole image and going from there.

masked_fruit_bgr = np.zeros_like(imported_fruit)


# I stole this from a google search that generated this code via AI
def is_in_range(number, low_high):
    """Returns True if number is within [start, stop), else False."""
    return number in range(low_high[0], low_high[1])


# imported_fruit.shape = (360, 661, 3)
def is_white(r, g, b):
    if r > 200 and b > 200 and g > 200:
        return True
    else:
        return False

def is_yellow(r, g, b):
    # I googled this criteria and got this from the AI generated google search
    if (r > 150) and (g > 150) and (b < 100) and (abs(int(r) - int(g)) < 100):
        return True
    else:
        return False

def is_green(r, g, b):
    return (int(g) > int(r) + 30) and (int(g) > int(b) + 30)
    

def is_pixel_green(r, g, b, min_intensity=100, ratio=1.0):
    r = int(r)
    g = int(g)
    b = int(b)

    return (g > min_intensity) and ((g > r * ratio) and (g > b * ratio))
    

for row in range(0, imported_fruit.shape[0]):
    for col in range(0, imported_fruit.shape[1]):
        b, g, r = imported_fruit[row, col]
        
        if (is_pixel_green(r, g, b, min_intensity=30)):
            masked_fruit_bgr[row, col, :] = imported_fruit[row, col, :]



# For HSV, we just need to find the range of green values for the hue. As such,
# I used my friend (google) to get this range. With this range, we then just
# check every point in the image for the range.

# The range I will use is defined as 80° to 140°.
hsv_fruit = cv2.cvtColor(imported_fruit, cv2.COLOR_BGR2HSV)

# I finally looked up the better way to implement masks compared to the prior approach of using for loops.
lower_green = np.array([30, 40, 40])
upper_green = np.array([90, 255, 255])

# Binary mask is created :D
mask_hsv_fruit = cv2.inRange(hsv_fruit, lower_green, upper_green)

masked_hsv_fruit = cv2.bitwise_and(hsv_fruit, hsv_fruit, mask=mask_hsv_fruit)

converted_bgr_to_rgb_masked_fruit = cv2.cvtColor(masked_fruit_bgr, cv2.COLOR_BGR2RGB)
converted_hsv_to_rgb_masked_fruit = cv2.cvtColor(masked_hsv_fruit, cv2.COLOR_HSV2RGB)

fig, ax = plt.subplots(1, 3)

ax[0].imshow(cv2.cvtColor(imported_fruit, cv2.COLOR_BGR2RGB))
ax[0].set_title("Original")

ax[1].imshow(converted_bgr_to_rgb_masked_fruit)
ax[1].set_title("RGB Masked")            

ax[2].imshow(converted_hsv_to_rgb_masked_fruit)
ax[2].set_title("HSV Masked")

plt.show()

print(imported_fruit[120, 100])

In [ ]:

########### TASK 2B ###########

# b) Import the "fruits.jpg" image 
#
# <img src = "Images/fruits.jpg" style="width:300px;height:180px">
#
# You want to extract the orange colors from the image above. Apply a suitable
# mask to extract the orange pixels in RGB color space. 
#
# You will now do the same thing in HSV space. Covert the same image to HSV
# space and apply a suitable mask to extract the orange color pixels.  

# Compared to my prior apporach, it will be better (probably not). That is because I am revisiting this after doing nearly all of the tasks.

imported_fruits = cv2.imread("Images/fruits.jpg")

def is_white(r, g, b):
    if r >= 200 and b >= 200 and g >= 200:
        if r == 255 and b == 255 and g == 255:
            return True

    return False

def is_pixel_orange(r, g, b, min_intensity=100, ratio=1.0):
    return (float((r + g) / 2) > min_intensity) and ((r > g * ratio) and (r > b * ratio))

rows, cols, _ = imported_fruits.shape

masked_orange_fruit_bgr = np.zeros_like(imported_fruits)

for row in range(0, rows):
    for col in range(0, cols):
        b, g, r = imported_fruits[row, col]

        r = int(r)
        g = int(g)
        b = int(b)

        if (is_white(r, g, b) and (r != b or r != g)) or is_pixel_orange(r, g, b, min_intensity=150, ratio=1.2):
            masked_orange_fruit_bgr[row, col] = imported_fruits[row, col]

print(imported_fruits[150, 300])


hsv_fruits = cv2.cvtColor(imported_fruits, cv2.COLOR_BGR2HSV)

# Googled these thresholds 
lower_orange = np.array([5, 150, 150])
upper_orange = np.array([20, 255, 255])

mask_hsv_fruits = cv2.inRange(hsv_fruits, lower_orange, upper_orange)
masked_hsv_fruits = cv2.bitwise_and(hsv_fruits, hsv_fruits, mask=mask_hsv_fruits)


fig, ax = plt.subplots(1, 3, figsize=(10,6))
ax[0].imshow(cv2.cvtColor(imported_fruits, cv2.COLOR_BGR2RGB)) 
ax[0].set_title("Original")
ax[1].imshow(cv2.cvtColor(masked_orange_fruit_bgr, cv2.COLOR_BGR2RGB))
ax[1].set_title("RGB Masked")
ax[2].imshow(cv2.cvtColor(masked_hsv_fruits, cv2.COLOR_HSV2RGB))
ax[2].set_title("HSV Masked")


# Display the original image, the masked image in RGB space and the masked image in HSV color space in a 1X3 plot. 

# Task 3 (20 points)

Import the "pumpkins.jpeg" image and apply the following point processing to it.
<img src = "Images/pumpkins.jpeg" style="width:250px;height:200px">

a) Create a darker image by subtracting 100 from each pixel intensity value.
DONE

b) Create a nonlinear lower contrast image by applying the formula $(x / 255)^{1/3}$ X 255, where x is the original pixel intensity.
DONE

c) Invert the original image where the pixel values are updated as 255 - current pixel intensity value.
DONE

d) Create a brighter image by adding 100 to each pixel intensity value.
DONE

e) Create a nonlinear higher contrast image by applying the formula $(x / 255)^{2}$ X 255, where x is the original pixel intensity.
DONE

Display all of these images along with the original image in a 2X3 grid.

In [ ]:

########### TASK 3A ###########

imported_pumpkins_bgr = cv2.imread("Images/pumpkins.jpeg")

# So to "subtract 100 from each pixel intensity value", I will interpret this
# as subtracting each channel, ensuring to apply clipping.

task3a_darkened_pumpkin = imported_pumpkins_bgr.copy()
task3a_darkened_pumpkin = cv2.subtract(imported_pumpkins_bgr, 100)

plt.imshow(cv2.cvtColor(task3a_darkened_pumpkin, cv2.COLOR_BGR2RGB))
plt.title("Task 3A")
plt.show()

In [ ]:

########### TASK 3B ###########

# Ok so prior I could use the cv2.subtract, but now I will do the per-pixel
# operation and apply the formula

# Formula: $(x / 255)^{1/3}$ X 255
task3b_nonlinear_image = imported_pumpkins_bgr.copy()

# imported_pumpkins_bgr.shape = (298, 400, 3)
#
# Addendum: I got pranked and learned that x is a normalized value. So I need to normalize first >:(
# Addendum 2: I was wrong, there is no such min-max normalization since the nonlinear_contrast_formula already is doing a form of normalization by dividing by 255
blue_range = [np.min(task3b_nonlinear_image[:, :, 0]), np.max(task3b_nonlinear_image[:, :, 0])]
green_range = [np.min(task3b_nonlinear_image[:, :, 1]), np.max(task3b_nonlinear_image[:, :, 1])]
red_range = [np.min(task3b_nonlinear_image[:, :, 2]), np.max(task3b_nonlinear_image[:, :, 2])]

# We will use min/max normalization

def nonlinear_contrast_formula(channel):
    return float((channel / 255)**(1/3) * 255)

for row in range(0, 298):
    for col in range(0, 400):
        b, g, r = task3b_nonlinear_image[row, col, :]
        task3b_nonlinear_image[row, col, :] = [nonlinear_contrast_formula(b), nonlinear_contrast_formula(g), nonlinear_contrast_formula(r)]

# So PLT doesn't actually need the rgb channels to be in the range 0-255 since
# it has its own normalization metrics, so I did not bother with making the
# channels back into 0-255

plt.imshow(cv2.cvtColor(task3b_nonlinear_image, cv2.COLOR_BGR2RGB))
plt.title("Task 3B")
plt.show()

In [ ]:

########### TASK 3C ###########

# So we just invert the image. I will also just use for loops again, there is probably a more elegant solution with lambda functions or something
task3c_inverted_image = imported_pumpkins_bgr.copy() 

for row in range(0, 297):
    for col in range(0, 399):
        b, g, r = task3c_inverted_image[row, col, :]
        b = 255 - b
        g = 255 - g 
        r = 255 - r 
        task3c_inverted_image[row, col, :] = [b, g, r]

plt.imshow(cv2.cvtColor(task3c_inverted_image, cv2.COLOR_BGR2RGB))
plt.title("Task 3C")
plt.show()

In [ ]:

########### TASK 3D ###########

# For this, do the same approach from task 3a

task3d_brighted_image = imported_pumpkins_bgr.copy()
task3d_brighted_image = cv2.add(task3d_brighted_image, 100)

plt.imshow(cv2.cvtColor(task3d_brighted_image, cv2.COLOR_BGR2RGB))
plt.title("Task 3D")
plt.show()

In [ ]:

########### TASK 3E ###########

# e) Create a nonlinear higher contrast image by applying the formula $(x / 255)^{2}$ X 255, where x is the original pixel intensity.
# Formula: $(x / 255)^{2}$ X 255
task3e_nonlinear_image = imported_pumpkins_bgr.copy()

# imported_pumpkins_bgr.shape = (298, 400, 3)
#
# Addendum: I got pranked and learned that x is a normalized value. So I need to normalize first >:(
blue_range = [np.min(task3e_nonlinear_image[:, :, 0]), np.max(task3e_nonlinear_image[:, :, 0])]
green_range = [np.min(task3e_nonlinear_image[:, :, 1]), np.max(task3e_nonlinear_image[:, :, 1])]
red_range = [np.min(task3e_nonlinear_image[:, :, 2]), np.max(task3e_nonlinear_image[:, :, 2])]

# We will use min/max normalization

def nonlinear_higher_contrast_formula(channel):
    return float((channel / 255)**(2) * 255)

for row in range(0, 298):
    for col in range(0, 400):
        b, g, r = task3e_nonlinear_image[row, col, :]
        task3e_nonlinear_image[row, col, :] = [nonlinear_higher_contrast_formula(b), nonlinear_higher_contrast_formula(g), nonlinear_higher_contrast_formula(r)]

plt.imshow(cv2.cvtColor(task3e_nonlinear_image, cv2.COLOR_BGR2RGB))
plt.title("Task 3e")
plt.show()

In [ ]:

# Do the display all them images in 2x3

fig, ax = plt.subplots(2, 3, figsize=(12,8))
ax[0, 0].imshow(cv2.cvtColor(imported_pumpkins_bgr, cv2.COLOR_BGR2RGB))
ax[0, 0].set_title("Original")

ax[0, 1].imshow(cv2.cvtColor(task3a_darkened_pumpkin, cv2.COLOR_BGR2RGB))
ax[0, 1].set_title("Task 3A")

ax[0, 2].imshow(cv2.cvtColor(task3b_nonlinear_image, cv2.COLOR_BGR2RGB))
ax[0, 2].set_title("Task 3B")

ax[1, 0].imshow(cv2.cvtColor(task3c_inverted_image, cv2.COLOR_BGR2RGB))
ax[1, 0].set_title("Task 3C")

ax[1, 1].imshow(cv2.cvtColor(task3d_brighted_image, cv2.COLOR_BGR2RGB))
ax[1, 1].set_title("Task 3D")

ax[1, 2].imshow(cv2.cvtColor(task3e_nonlinear_image, cv2.COLOR_BGR2RGB))
ax[1, 2].set_title("Task 3E")

plt.show()

### Task 4 (30 points) 

Import the "butterfly.jpg" image 
<img src = "Images/butterfly.jpg" style="width:300px;height:200px">

a) Perform average blur on the imported image with kernel size 5X5 and 11X11.
Display the original image with the filtered/processed images in a 1X3 grid.
DONE


b) Add salt and pepper noise as 10% of all pixels. Perform gaussian blur with
kernel size 5X5, sigma (standard deviation of the Gaussian) of 2.0 and
another gaussian blur with kernel size 11X11, sigma 5. Display the image with
salt and pepper noise and the filtered/processed images in a 1X3 grid.
DONE


c) Add salt and pepper noise as 20% of all pixels. Perform median blur with
kernel size 5X5 and 11X11. Display the image with salt and pepper noise and
the filtered/processed images in a 1X3 grid.
DONE


d) Resize the original image to 50X50 pixels. Also, resize the guassian
blurred image with kernel size 7X7 and sigma 4.0 to 50X50 pixels. Display the
original image with the resized images in a 1X3 grid.
DONE

e) Generate a 3 level Gaussian pyramid (L0 - original image, L1, L2, L3) by
choosing an appropriate kernel size and display the images in a 1X4 grid.
Check the slide (03-Subsampling-and-Image-Pyramid.pdf) for reference.
DONE

f) Generate a 3 level Laplacian pyramid (L0 - original image, L1, L2, L3) by
choosing an appropriate kernel size and display the images in a 1X3 grid.
Check the slide (03-Subsampling-and-Image-Pyramid.pdf) for reference.
DONE

Note: Apply zero-padding to make the filtered image size same as original image. 

In [ ]:
imported_butterfly = cv2.imread("Images/butterfly.jpg")

In [ ]:

########### TASK 4A ###########

# So first we gotta make a padded image for the respective kernels. I can use
# the power of math (padding 4 and padding 10) and just add new rows with
# respective BGR values that are 0s. Then use a for loop and start the the
# correct row and col, end at the correct row and col, and them apply the
# correlation operation to for the respective kernel. I hope to find some cool
# NP implementations of this...

# So there exists cv2.filter2D so this should be good for what I'm doing.
average_filter5x5 = np.full((5,5), float(1/25))
average_filter11x11 = np.full((11,11), float(1/121))

butterfly_5x5_average_blur = cv2.filter2D(imported_butterfly, -1, average_filter5x5) 
butterfly_11x11_average_blur = cv2.filter2D(imported_butterfly, -1, average_filter11x11) 
# the -1 is a mysterious parameter for the ddepth. It has something to do with
# having the dimensionality, the -1 parameter ensures that the shape doesn't
# change. So it likely has padding in the implementations.

rgb_butterfly = cv2.cvtColor(imported_butterfly, cv2.COLOR_BGR2RGB)
rgb_butterfly_5x5_average_blur = cv2.cvtColor(butterfly_5x5_average_blur, cv2.COLOR_BGR2RGB)
rgb_butterfly_11x11_average_blur = cv2.cvtColor(butterfly_11x11_average_blur, cv2.COLOR_BGR2RGB)

fig, ax = plt.subplots(1, 3, figsize=(15,3))
ax[0].imshow(rgb_butterfly)
ax[0].set_title("Original")
ax[1].imshow(rgb_butterfly_5x5_average_blur)
ax[1].set_title("5x5")
ax[2].imshow(rgb_butterfly_11x11_average_blur)
ax[2].set_title("11x11")

plt.show()

In [ ]:

########### TASK 4B ###########

# To add salt and pepper, I used the code from this website: https://gist.github.com/gutierrezps/f4ddad3bbd2ad5a9b96e3c06378e28b4

def sp_noise(image, prob):
    '''
    Add salt and pepper noise to image
    prob: Probability of the noise
    '''
    output = image.copy()
    if len(image.shape) == 2:
        black = 0
        white = 255            
    else:
        colorspace = image.shape[2]
        if colorspace == 3:  # RGB
            black = np.array([0, 0, 0], dtype='uint8')
            white = np.array([255, 255, 255], dtype='uint8')
        else:  # RGBA
            black = np.array([0, 0, 0, 255], dtype='uint8')
            white = np.array([255, 255, 255, 255], dtype='uint8')
    probs = np.random.random(output.shape[:2])
    output[probs < (prob / 2)] = black
    output[probs > 1 - (prob / 2)] = white
    return output

# I like this implementation since it uses np.random to make the noise and then filters based on the probablity. Thus, I can implement the "10%" and "20%" salting :D

noisy_butterfly_10 = sp_noise(imported_butterfly, .1)
noisy_guassian_butterfly_10_kernel_5x5_sigma_2 = cv2.GaussianBlur(noisy_butterfly_10, (5,5), 2.0)
noisy_guassian_butterfly_10_kernel_11x11_sigma_5 = cv2.GaussianBlur(noisy_butterfly_10, (11,11), 5.0)

fig, ax = plt.subplots(1, 3, figsize=(15,3))
ax[0].imshow(noisy_butterfly_10)
ax[0].set_title("Salt and Pepper 10%")
ax[1].imshow(noisy_guassian_butterfly_10_kernel_5x5_sigma_2)
ax[1].set_title("Guassian_kernel5x5_sigma2")
ax[2].imshow(noisy_guassian_butterfly_10_kernel_11x11_sigma_5)
ax[2].set_title("Guassian_kernel11x11_sigma5")

plt.show()

# b) Add salt and pepper noise as 10% of all pixels. Perform gaussian blur with
# kernel size 5X5, sigma (standard deviation of the Gaussian) of 2.0 and
# another gaussian blur with kernel size 11X11, sigma 5. Display the image with
# salt and pepper noise and the filtered/processed images in a 1X3 grid.

In [ ]:

########### TASK 4C ###########

# c) Add salt and pepper noise as 20% of all pixels. Perform median blur with
# kernel size 5X5 and 11X11. Display the image with salt and pepper noise and
# the filtered/processed images in a 1X3 grid.

noisy_butterfly_20 = sp_noise(imported_butterfly, .2)
noisy_median_butterfly_20_kernel_5x5 = cv2.medianBlur(noisy_butterfly_20, 5)
noisy_median_butterfly_20_kernel_11x11 = cv2.medianBlur(noisy_butterfly_20, 11)

fig, ax = plt.subplots(1, 3, figsize=(15,3))
ax[0].imshow(noisy_butterfly_20)
ax[0].set_title("Salt and Pepper 20%")
ax[1].imshow(noisy_median_butterfly_20_kernel_5x5)
ax[1].set_title("noisy_median_butterfly_20_kernel_5x5")
ax[2].imshow(noisy_median_butterfly_20_kernel_11x11)
ax[2].set_title("noisy_median_butterfly_20_kernel_11x11")

plt.show()

In [ ]:

########### TASK 4D ###########

# d) Resize the original image to 50X50 pixels. Also, resize the guassian
# blurred image with kernel size 7X7 and sigma 4.0 to 50X50 pixels. Display the
# original image with the resized images in a 1X3 grid.

# I need to calculate the row_indices and col_indices and then use np.delete to remove the indices. 

def resize_to_50x50(image):
    rows, cols, _ = image.shape

    row_indices = np.linspace(0, rows - 1, 50).astype(int)
    col_indices = np.linspace(1, cols - 1, 50).astype(int)


    resized_image = image[row_indices][: , col_indices]


    print(f"Length of rows_indices: {len(row_indices)}")
    print(f"Shape of resized image: {resized_image.shape}")

    return resized_image


resized_butterfly_50x50 = resize_to_50x50(imported_butterfly)
resized_butterfly_50x50_guassian = resize_to_50x50(cv2.GaussianBlur(imported_butterfly, (7,7), 4.0))

fig, ax = plt.subplots(1, 3, figsize=(15,3))
ax[0].imshow(imported_butterfly)
ax[0].set_title("Original")
ax[1].imshow(resized_butterfly_50x50)
ax[1].set_title("resized_butterfly_50x50")
ax[2].imshow(resized_butterfly_50x50_guassian)
ax[2].set_title("resized_butterfly_50x50_guassian")

plt.show()

In [ ]:

########### TASK 4E ###########

# e) Generate a 3 level Gaussian pyramid (L0 - original image, L1, L2, L3) by
# choosing an appropriate kernel size and display the images in a 1X4 grid.
# Check the slide (03-Subsampling-and-Image-Pyramid.pdf) for reference.

# I need to repurpose the function from the last exercise but this time change the constants to 2
def resize_to_half(image):
    rows, cols, _ = image.shape


    row_indices = np.linspace(0, rows - 1, int(rows/2)).astype(int)
    col_indices = np.linspace(1, cols - 1, int(cols/2)).astype(int)


    resized_image = image[row_indices][: , col_indices]


    print(f"Length of rows_indices: {len(row_indices)}")
    print(f"Shape of resized image: {resized_image.shape}")

    return resized_image

L0 = imported_butterfly.copy()
L1 = resize_to_half(cv2.GaussianBlur(L0, (5,5), 0))
L2 = resize_to_half(cv2.GaussianBlur(L1, (5,5), 0))
L3 = resize_to_half(cv2.GaussianBlur(L2, (5,5), 0))

fix, ax = plt.subplots(1, 4, figsize=(15,3))
ax[0].imshow(L0)
ax[0].set_title("L0")
ax[1].imshow(L1)
ax[1].set_title("L1")
ax[2].imshow(L2)
ax[2].set_title("L2")
ax[3].imshow(L3)
ax[3].set_title("L3")

plt.show()

In [ ]:

########### TASK 4f ###########

# f) Generate a 3 level Laplacian pyramid (L0 - original image, L1, L2, L3) by
# choosing an appropriate kernel size and display the images in a 1X3 grid.
# Check the slide (03-Subsampling-and-Image-Pyramid.pdf) for reference.

# ChatGPT told me that Laplacian is formed by this formula which is based on the expansion of the previous layers of the gaussian pyramid steps.imported_butterfly
# Laplacian = Gaussian_i - Expand(Gaussian_i+1)

L1_expanded = cv2.resize(L1, (L0.shape[1], L0.shape[0])) # apparently resize and numpy use opposite ordering conventions 
L2_expanded = cv2.resize(L2, (L1.shape[1], L1.shape[0]))
L3_expanded = cv2.resize(L3, (L2.shape[1], L2.shape[0]))

L0_lap = L0.astype(float) - L1_expanded.astype(float)
L1_lap = L1.astype(float) - L2_expanded.astype(float) 
L2_lap = L2.astype(float) - L3_expanded.astype(float) 
L3_lap = L3 


fig, ax = plt.subplots(1, 3, figsize=(15,3))
ax[0].imshow(L0_lap)
ax[0].set_title("L0 Laplacian")
ax[1].imshow(L1_lap)
ax[1].set_title("L1 Laplacian")
ax[2].imshow(L2_lap)
ax[2].set_title("L2 Laplacian")
#ax[3].imshow(L3_lap)
#ax[3].set_title("L3 Laplacian")

# The L0 is misleading since the first layer of the laplacian is the difference between the enlarged guassian and the previous image, so we should only have 3 figures to show here.

plt.show()

### Task 5 (15 points) 

a) Import the "uta.png" image and apply the "shadow effect" on the image. (apply an appropriate amount of blurring and you may shift the gaussian blurred image to create the shadow effect)

<img src = "Images/uta.png" style="width:360px;height:140px">

b) Import the "grand_teton.jpg" image and apply the "tilt shift effect" on the image where the barn and the mountains should be in focus and the background and foreground should be blurred (any reasonable amount of blurring of the foreground and backgroud will be considered as correct)

<img src = "Images/grand_teton.jpg" style="width:400px;height:200px">

In [ ]:

########### TASK 5A ###########

# a) Import the "uta.png" image and apply the "shadow effect" on the image.
# (apply an appropriate amount of blurring and you may shift the gaussian
# blurred image to create the shadow effect)

imported_uta_rgb = cv2.cvtColor(cv2.imread("Images/UTA.png"), cv2.COLOR_BGR2RGB)

# Mask UTA -> Apply Blur -> Shift
uta_hsv = cv2.cvtColor(imported_uta_rgb, cv2.COLOR_RGB2HSV)

lower_threshold = np.array([0, 0, 0])
upper_threshold = np.array([180, 255, 50])

uta_mask = cv2.inRange(uta_hsv, lower_threshold, upper_threshold)
cropped_uta = cv2.bitwise_and(uta_hsv, uta_hsv, mask=uta_mask)

cropped_uta_rgb = cv2.cvtColor(cropped_uta, cv2.COLOR_HSV2RGB)

blurred_uta = cv2.GaussianBlur(cropped_uta_rgb, (5, 5), 5.0)

# Shift Mask
tx, ty = 2, 2
M = np.float32([[1, 0, tx],[0, 1, ty]]) 

shifted_blurred_uta = cv2.warpAffine(blurred_uta, M, (imported_uta_rgb.shape[1], imported_uta_rgb.shape[0]))

# I'm about to do the most inelegant approach to add the blurred photo
rows, cols, _ = imported_uta_rgb.shape

canvas = imported_uta_rgb.copy()

def is_white(r, g, b):
    if r > 200 and g > 200 and b > 200:
        return True
    else:
        return False 

for row in range(0, rows):
    for col in range(0, cols):

        b, g, r = imported_uta_rgb[row, col]

        if is_white(r, g, b) and not np.array_equal(shifted_blurred_uta[row, col], [0, 0, 0]):
            canvas[row, col, :] = shifted_blurred_uta[row, col, :]

plt.imshow(canvas)
plt.title("Cool Shadow Effect")
plt.show()
            

In [ ]:

########### TASK 5b ###########

# b) Import the "grand_teton.jpg" image and apply the "tilt shift effect" on
# the image where the barn and the mountains should be in focus and the
# background and foreground should be blurred (any reasonable amount of
# blurring of the foreground and backgroud will be considered as correct)

imported_teton_rgb = cv2.cvtColor(cv2.imread("Images/grand_teton.jpg"), cv2.COLOR_BGR2RGB)

# Define ROI via corner coordinates 
top_left_corner = [100, 400]
bottom_right_corner = [300, 750]

# Blur image 
blurred_teton_rgb = cv2.GaussianBlur(imported_teton_rgb, (5,5), 3.0)

canvas = blurred_teton_rgb.copy()

for row in range(0, canvas.shape[0]):
    for col in range(0, canvas.shape[1]):
        if row > top_left_corner[0] and row < bottom_right_corner[0] and col > top_left_corner[1] and col < bottom_right_corner[1]:
            canvas[row, col, :] = imported_teton_rgb[row, col, :]

plt.imshow(canvas)
plt.title("Cool Focus Effect")
plt.show()

## Submission Guidelines:

1. Submit through Canvas your source code in a single .ipynb file. The name of the .ipynb file should be YourStudentID.ipynb. (For example: 1001234567.ipynb)

2. Import all the images from the ./Images directory. Your TA will use the same directory name to grade your submission.

3. You don't need to attach the image file with your submission.